<a href="https://colab.research.google.com/github/tejalkv0/CrewAI/blob/main/CrewAI_and_LLM_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.4/232.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/

In [12]:
import os, time, calendar
from datetime import datetime
from getpass import getpass
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool

# ---- 1. LLM setup (free Gemini key from https://aistudio.google.com/apikey) ----
os.environ["GEMINI_API_KEY"] = getpass("Gen")
llm = LLM(model="gemini/gemini-3.5-flash")   # change the model name if needed

# ---- 2. Shared storage for what the agent creates ----
alarms, reminders = [], []

# ---- 3. Tools the agent can use ----
@tool("set_alarm")
def set_alarm(time_24h: str) -> str:
    """Set an alarm. Input is time in 24-hour HH:MM format, e.g. '12:00'."""
    alarms.append(time_24h)
    return f"Alarm set for {time_24h}"

@tool("set_reminder")
def set_reminder(text_and_time: str) -> str:
    """Set a reminder. Input format: 'reminder text | HH:MM' e.g. 'Lunch time | 12:00'."""
    text, t = [x.strip() for x in text_and_time.split("|")]
    reminders.append((t, text))
    return f"Reminder '{text}' set for {t}"

@tool("show_calendar")
def show_calendar(dummy: str = "") -> str:
    """Show the current month's calendar."""
    now = datetime.now()
    return calendar.month(now.year, now.month) + f"\nToday: {now:%d %b %Y}"

# ---- 4. Agent ----
assistant = Agent(
    role="Personal Scheduler",
    goal="Manage alarms, reminders and the calendar for the user",
    backstory="You are a helpful assistant that uses tools to schedule things.",
    tools=[set_alarm, set_reminder, show_calendar],
    llm=llm,
    verbose=True,
)

# ---- 5. Tasks (these are the instructions fed to the LLM) ----
task1 = Task(
    description="Set an alarm for 12:00 (24-hour format).",
    expected_output="Confirmation that the alarm is set.",
    agent=assistant,
)
task2 = Task(
    description="Set a reminder with text 'Lunch time' for 12:00.",
    expected_output="Confirmation that the reminder is set.",
    agent=assistant,
)
task3 = Task(
    description="Display the current month's calendar to the user.",
    expected_output="The calendar of the current month.",
    agent=assistant,
)

crew = Crew(agents=[assistant], tasks=[task1, task2, task3])

result = await crew.kickoff_async()


print("\n===== FINAL OUTPUT =====")
print(result)
print("\nAlarms:", alarms)
print("Reminders:", reminders)

Gen··········


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Scheduler                                                                                      │
│                                                                                                                 │
│  Task: Set an alarm for 12:00 (24-hour format).                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool set_alarm executed with result: Alarm set for 12:00...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Scheduler                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I have successfully set an alarm for you at 12:00.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Scheduler                                                                                      │
│                                                                                                                 │
│  Task: Set a reminder with text 'Lunch time' for 12:00.                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Scheduler                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I have successfully set a reminder for 'Lunch time' at 12:00.                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Scheduler                                                                                      │
│                                                                                                                 │
│  Task: Display the current month's calendar to the user.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool show_calendar executed with result:     October 2026
Mo Tu We Th Fr Sa Su
          1  2  3  4
 5  6  7  8  9 10 11
12 13 14 15 16 17 18
19 20 21 22 23 24 25
26 27 28 29 30 31

Today: 09 Oct 2026...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Scheduler                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│      October 2026                                                                                               │
│  Mo Tu We Th Fr Sa Su                                                                                           │
│            1  2  3  4                                                                                           │
│   5  6  7  8  9 10 11                                                                                           │
│  12 13 14 15 16 17 18                                                                                           │
│  19 20 21 22 23 24 25                                                                                           │
│  26 27 28 29 30 31                                                                                              │
│                                                                                                                 │
│  Today: 09 Oct 2026                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== FINAL OUTPUT =====
    October 2026
Mo Tu We Th Fr Sa Su
          1  2  3  4
 5  6  7  8  9 10 11
12 13 14 15 16 17 18
19 20 21 22 23 24 25
26 27 28 29 30 31

Today: 09 Oct 2026

Alarms: ['12:00']
Reminders: [('12:00', 'Lunch time')]


In [17]:
alarms.append("12:00")
print(alarms, reminders)

['12:00'] [('12:00', 'Lunch time')]


In [18]:
# ---------- Part A: plain Python checks (ground truth) ----------
now = datetime.now()
cal_text = calendar.month(now.year, now.month)

checks = {
    "Alarm set for 12:00": "12:00" in alarms,
    "Lunch reminder set for 12:00": any(t == "12:00" and "lunch" in text.lower() for t, text in reminders),
    "Calendar shows current month": calendar.month_name[now.month] in cal_text and str(now.year) in cal_text,
}

print("===== PYTHON CHECKS =====")
for name, ok in checks.items():
    print(("✅ PASS" if ok else "❌ FAIL"), "-", name)


# ---------- Part B: LLM tester agent ----------
@tool("get_scheduler_state")
def get_scheduler_state(dummy: str = "") -> str:
    """Returns the current alarms, reminders and calendar so they can be verified."""
    return (
        f"Alarms: {alarms}\n"
        f"Reminders (time, text): {reminders}\n"
        f"Calendar:\n{cal_text}\n"
        f"Today: {now:%d %b %Y}"
    )

tester = Agent(
    role="QA Tester",
    goal="Verify that the alarm, reminder and calendar were created correctly",
    backstory="You are a strict tester. You only trust data returned by your tool and never assume.",
    tools=[get_scheduler_state],
    llm=llm,
    verbose=True,
)

verify_task = Task(
    description=(
        "Call get_scheduler_state, then verify:\n"
        "1. An alarm exists for 12:00.\n"
        "2. A lunch reminder exists for 12:00.\n"
        "3. The calendar belongs to the current month and year.\n"
        "Give PASS or FAIL for each check with a one-line reason, then an overall verdict."
    ),
    expected_output="A short test report with PASS/FAIL for each of the 3 checks and an overall verdict.",
    agent=tester,
)

test_crew = Crew(agents=[tester], tasks=[verify_task])
report = await test_crew.kickoff_async()

print("\n===== LLM TEST REPORT =====")
print(report)

===== PYTHON CHECKS =====
✅ PASS - Alarm set for 12:00
✅ PASS - Lunch reminder set for 12:00
✅ PASS - Calendar shows current month


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Tester                                                                                               │
│                                                                                                                 │
│  Task: Call get_scheduler_state, then verify:                                                                   │
│  1. An alarm exists for 12:00.                                                                                  │
│  2. A lunch reminder exists for 12:00.                                                                          │
│  3. The calendar belongs to the current month and year.                                                         │
│  Give PASS or FAIL for each check with a one-line reason, then an overall verdict.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool get_scheduler_state executed with result: Alarms: ['12:00']
Reminders (time, text): [('12:00', 'Lunch time')]
Calendar:
    October 2026
Mo Tu We Th Fr Sa Su
          1  2  3  4
 5  6  7  8  9 10 11
12 13 14 15 16 17 18
19 20 21 22 23 24 25
...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Tester                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Test Report:**                                                                                               │
│                                                                                                                 │
│  1. **Alarm Check**: PASS                                                                                       │
│     - Reason: The alarm list successfully contains an alarm set for '12:00'.                                    │
│  2. **Lunch Reminder Check**: PASS                                                                              │
│     - Reason: A reminder for "Lunch time" is successfully set for '12:00'.                                      │
│  3. **Calendar Check**: PASS                                                                                    │
│     - Reason: The calendar displays October 2026, which matches the current month and year of the system        │
│  (October 2026).                                                                                                │
│                                                                                                                 │
│  **Overall Verdict**: PASS                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== LLM TEST REPORT =====
**Test Report:**

1. **Alarm Check**: PASS
   - Reason: The alarm list successfully contains an alarm set for '12:00'.
2. **Lunch Reminder Check**: PASS
   - Reason: A reminder for "Lunch time" is successfully set for '12:00'.
3. **Calendar Check**: PASS
   - Reason: The calendar displays October 2026, which matches the current month and year of the system (October 2026).

**Overall Verdict**: PASS
